# Silver Layer — Cleaning, Deduplication & Fraud Detection

This notebook reads raw Bronze tables from `workspace.bronze_schema` and writes cleaned, typed,
and fraud-flagged Delta tables to `workspace.silver_schema`.

**Key transformations:**
1. **Deduplication** on composite key (`LocationId`, `BusinessDate`, `Id`)
2. **Column cleanup** — drop all-null columns, fill numeric nulls (< 5% missing) with 0
3. **Type casting** — dates, timestamps, monetary values
4. **Fraud detection** — 6 hypotheses with boolean flags + suspect_fraud_checks table
5. **Data quality** — Delta constraints on primary keys + quarantine table for bad data

In [0]:
# ============================================================================
# Silver Layer — Imports, Configuration & Helper Functions
# ============================================================================
from pyspark.sql.functions import (
    col, lit, when, size, hour, to_date, to_timestamp, trim, expr,
    sum as _sum, count as _count, explode, collect_list, array, array_intersect,
    percentile_approx, rank, desc, broadcast
)
from pyspark.sql.types import (
    DateType, TimestampType, DoubleType, DecimalType,
    NumericType, ArrayType, MapType, StringType
)
from pyspark.sql.window import Window

# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------
BRONZE_SCHEMA = "workspace.bronze_schema"
SILVER_SCHEMA = "workspace.silver_schema"

# Per-table type-casting rules: {table_name: {column_name: "target_sql_type"}}
# These are applied in addition to the common string-trim and all-null-column drops.
TYPE_CAST_RULES = {
    "checks": {
        "BusinessDate":  "DATE",
        "OpenTime":      "TIMESTAMP",
        "CloseTime":     "TIMESTAMP",
        "ServiceCharge": "DOUBLE",        # monetary field currently stored as LONG
    },
    "employees": {
        "DateOfBirth": "DATE",
        "HireDate":    "DATE",
    },
    "locations_with_coordinates": {
        "OpenDate":   "DATE",
        "ClosedDate": "DATE",
    },
    "checks_items": {
        "OrderTime": "TIMESTAMP",
    },
}

# ------------------------------------------------------------------
# Create the Silver schema (idempotent)
# ------------------------------------------------------------------
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
print(f"✅ Schema '{SILVER_SCHEMA}' is ready.")

# ------------------------------------------------------------------
# Helper: drop columns that are 100 % NULL
# ------------------------------------------------------------------
def drop_all_null_columns(df, table_name=""):
    """Remove columns where every row is NULL."""
    total = df.count()
    if total == 0:
        return df
    null_cols = []
    for c in df.columns:
        if df.filter(col(c).isNull()).count() == total:
            null_cols.append(c)
    if null_cols:
        print(f"  [{table_name}] Dropping all-null columns: {null_cols}")
        df = df.drop(*null_cols)
    return df

# ------------------------------------------------------------------
# Helper: fill scalar-numeric nulls with 0 when missing rate < 5 %
# ------------------------------------------------------------------
def fill_numeric_nulls(df, table_name="", threshold=0.05):
    """Fillna(0) on scalar numeric columns whose NULL rate is below threshold."""
    total = df.count()
    if total == 0:
        return df
    fill_map = {}
    for field in df.schema.fields:
        if isinstance(field.dataType, (ArrayType, MapType)):
            continue                       # skip arrays / maps
        if not isinstance(field.dataType, NumericType):
            continue                       # skip non-numeric
        null_pct = df.filter(col(field.name).isNull()).count() / total
        if 0 < null_pct < threshold:
            fill_map[field.name] = 0
    if fill_map:
        print(f"  [{table_name}] Filling NULLs with 0: {list(fill_map.keys())}")
        df = df.fillna(fill_map)
    return df

# ------------------------------------------------------------------
# Helper: trim whitespace from every string column
# ------------------------------------------------------------------
def trim_string_columns(df):
    """Apply trim() to all StringType columns."""
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(field.name, trim(col(field.name)))
    return df

# ------------------------------------------------------------------
# Helper: apply per-table type casts defined in TYPE_CAST_RULES
# ------------------------------------------------------------------
def apply_type_casts(df, table_name):
    """Cast columns to the target types specified in TYPE_CAST_RULES."""
    rules = TYPE_CAST_RULES.get(table_name, {})
    for col_name, target_type in rules.items():
        if col_name in df.columns:
            df = df.withColumn(col_name, col(col_name).cast(target_type))
    return df

print("✅ Helpers defined: drop_all_null_columns, fill_numeric_nulls, trim_string_columns, apply_type_casts")

In [0]:
# ============================================================================
# 1. PROCESS THE MAIN `checks` TABLE
# ============================================================================
# The checks table is the core fact table. It receives:
#   • Quarantine of records with NULL primary keys
#   • Deduplication on (LocationId, BusinessDate, Id)
#   • Type casting (dates, timestamps, monetary)
#   • Derived aggregate columns from nested arrays
#   • Fraud-detection boolean flags (hypotheses a-e)
# Hypothesis (f) is added in the next cell after employee aggregation.
# ============================================================================

checks_bronze = spark.table(f"{BRONZE_SCHEMA}.checks")
bronze_count = checks_bronze.count()
print(f"▶ Reading {BRONZE_SCHEMA}.checks — {bronze_count:,} raw rows")

# ── 1a. Quarantine records with NULL primary-key columns ────────────────
# Primary composite key: (LocationId, BusinessDate, Id).
# Any record missing one of these is structurally invalid and cannot be
# deduplicated or joined downstream.  Send to quarantine table.
quarantine_df = checks_bronze.filter(
    col("LocationId").isNull() |
    col("BusinessDate").isNull() |
    col("Id").isNull()
)
quarantine_count = quarantine_df.count()
if quarantine_count > 0:
    (quarantine_df.write.format("delta").mode("overwrite")
     .saveAsTable(f"{SILVER_SCHEMA}.quarantine_bad_data"))
    print(f"  ⚠  Quarantined {quarantine_count:,} rows with NULL primary keys → {SILVER_SCHEMA}.quarantine_bad_data")
else:
    # ensure table exists even if empty so downstream code doesn't break
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.quarantine_bad_data
        AS SELECT * FROM {BRONZE_SCHEMA}.checks WHERE 1 = 0
    """)
    print("  ✅ No records with NULL primary keys found.")

# ── 1b. Keep only valid records for silver processing ─────────────────
checks_df = checks_bronze.filter(
    col("LocationId").isNotNull() &
    col("BusinessDate").isNotNull() &
    col("Id").isNotNull()
)

# ── 1c. Deduplicate on composite key (LocationId, BusinessDate, Id) ────
# Id (Check ID) is local to each restaurant location and date, so the
# triple (LocationId, BusinessDate, Id) uniquely identifies a check.
before_dedup = checks_df.count()
checks_df = checks_df.dropDuplicates(["LocationId", "BusinessDate", "Id"])
after_dedup = checks_df.count()
print(f"  Dedup: {before_dedup:,} → {after_dedup:,} rows (removed {before_dedup - after_dedup:,} duplicates)")

# ── 1d. Type casting ──────────────────────────────────────────────────
# BusinessDate  → DATE
# OpenTime      → TIMESTAMP  (format: yyyy-MM-dd'T'HH:mm)
# CloseTime     → TIMESTAMP
# ServiceCharge → DOUBLE     (monetary value currently stored as LONG)
checks_df = apply_type_casts(checks_df, "checks")

# ── 1e. Trim whitespace from string columns ───────────────────────────
checks_df = trim_string_columns(checks_df)

# ── 1f. Derive aggregate columns from nested arrays ───────────────────
# These scalar columns power the fraud-detection flags below and make
# downstream Silver/Gold queries much simpler.
checks_df = (checks_df
    # Total discount amount: sum of Discounts[].Amount
    .withColumn("total_discount_amount",
        expr("aggregate(transform(coalesce(Discounts, array()), x -> coalesce(x.Amount, 0.0)), 0.0D, (acc, x) -> acc + x)"))
    # Total payment amount: sum of Payments[].Amount
    .withColumn("total_payment_amount",
        expr("aggregate(transform(coalesce(Payments, array()), x -> coalesce(x.Amount, 0.0)), 0.0D, (acc, x) -> acc + x)"))
    # Total gratuity (tips): sum of Payments[].Gratuity
    .withColumn("total_gratuity",
        expr("aggregate(transform(coalesce(Payments, array()), x -> coalesce(x.Gratuity, 0.0)), 0.0D, (acc, x) -> acc + x)"))
    # Total void amount: sum of Voids[].Amount
    .withColumn("total_void_amount",
        expr("aggregate(transform(coalesce(Voids, array()), x -> coalesce(x.Amount, 0.0)), 0.0D, (acc, x) -> acc + x)"))
    # Number of line items on the check
    .withColumn("item_count", expr("size(coalesce(ItemsSold, array()))"))
    # Boolean convenience flags
    .withColumn("has_voids",     expr("size(coalesce(Voids, array())) > 0"))
    .withColumn("has_discounts", expr("size(coalesce(Discounts, array())) > 0"))
    # Check duration in minutes (CloseTime − OpenTime)
    .withColumn("check_duration_minutes",
        (col("CloseTime").cast("long") - col("OpenTime").cast("long")) / 60.0)
    # Hour-of-day the check was opened (for irregular-hours detection)
    .withColumn("open_hour", hour(col("OpenTime")))
    # Boolean: any item has a SIGNIFICANTLY negative SoldPrice (< -$5).
    # Small negatives (e.g. -$0.50 for a removed topping or partial comp)
    # are routine in restaurant POS systems and excluded; values below
    # -$5 more likely indicate suspicious price manipulation.
    .withColumn("has_negative_item_price",
        expr("exists(coalesce(ItemsSold, array()), x -> x.SoldPrice < -5.0)"))
    # Boolean: any payment has a SIGNIFICANTLY negative Gratuity (< -$5).
    # Small negatives (e.g. -$0.50 adjustments, -$1.00 tip refunds) are
    # routine in restaurant POS data (48 % of checks have some negative
    # tip at < 0); values below -$5 more likely indicate manipulation.
    .withColumn("has_negative_gratuity",
        expr("exists(coalesce(Payments, array()), x -> x.Gratuity < -5.0)"))
)

# Discount-to-payment ratio (guard against division by zero)
checks_df = checks_df.withColumn(
    "discount_ratio",
    when(col("total_payment_amount") > 0,
         col("total_discount_amount") / col("total_payment_amount")
    ).otherwise(lit(0.0))
)

print(f"  Derived 12 aggregate columns from nested arrays.")

# ── 1g. Fraud-detection boolean flags (hypotheses a-e) ─────────────────
# Each flag is a boolean column on the main checks table.  Records that
# trigger ANY flag are later copied to suspect_fraud_checks.

# (a) High-value checks with excessive discounts
#     Payment > $100 AND discount ratio > 25 %
checks_df = checks_df.withColumn(
    "is_fraud_high_value_excess_discount",
    (col("total_payment_amount") > 100.0) & (col("discount_ratio") > 0.25)
)

# (b) Low-value checks with excessive discounts
#     Payment < $20 AND discount ratio > 50 %
checks_df = checks_df.withColumn(
    "is_fraud_low_value_excess_discount",
    (col("total_payment_amount") < 20.0) & (col("discount_ratio") > 0.50)
)

# (c) In-restaurant checks with low duration, high traffic, voids/discounts
#     RevenueCenterId = 1 (In-Store), duration < 10 min, TrafficCount > 3,
#     and (has voids OR has discounts)
checks_df = checks_df.withColumn(
    "is_fraud_suspicious_in_store",
    (col("RevenueCenterId") == 1) &
    (col("check_duration_minutes") < 10) &
    (col("TrafficCount") > 3) &
    (col("has_voids") | col("has_discounts"))
)

# (d) Negative tips and/or significantly negative prices
#     Any payment has Gratuity < -$5, OR ServiceCharge < 0,
#     OR any item has SoldPrice < -$5 (refined: excludes routine
#     small comps/voids/tip-adjustments that are normal in POS data)
checks_df = checks_df.withColumn(
    "is_fraud_negative_values",
    col("has_negative_gratuity") |
    (col("ServiceCharge") < 0) |
    col("has_negative_item_price")
)

# (e) Irregular opening hours (2:00 AM – 7:00 AM)
checks_df = checks_df.withColumn(
    "is_fraud_irregular_hours",
    (col("open_hour") >= 2) & (col("open_hour") < 7)
)

print("  Added 5 fraud-detection flags (a-e). Flag (f) added in next cell.")

# ── 1h. Column cleanup ─────────────────────────────────────────────────
checks_df = drop_all_null_columns(checks_df, "checks")
checks_df = fill_numeric_nulls(checks_df, "checks")

print(f"  checks Silver DF ready: {checks_df.count():,} rows × {len(checks_df.columns)} columns")

In [0]:
# ============================================================================
# 2. EMPLOYEE FRAUD AGGREGATION (Hypothesis f) + WRITE CHECKS TO SILVER
# ============================================================================
# (f) Summary flags for employees with the highest volume or total value
#     of discounts applied.
# Approach:
#   1. Explode the Discounts array from the raw bronze checks table.
#   2. Aggregate total discount amount and discount count per EmployeeId.
#   3. Save the per-employee summary as a Silver table.
#   4. Identify employees in the top 5 % by total discount amount
#      (95th-percentile threshold).
#   5. Add a boolean flag `is_fraud_top_discount_employee` to every check
#      where any discount was applied by a top-5 % employee.
# ============================================================================

# ── 2a. Explode Discounts and aggregate per employee ─────────────────────
emp_discounts = (checks_bronze
    .select(explode(col("Discounts")).alias("d"))
    .filter(col("d.EmployeeId").isNotNull())
    .groupBy(col("d.EmployeeId").alias("EmployeeId"))
    .agg(
        _sum("d.Amount").alias("total_discount_amount"),
        _count(lit(1)).alias("discount_count")
    )
    .orderBy(desc("total_discount_amount"))
)

# Save the employee discount summary as its own Silver table
(emp_discounts.write.format("delta").mode("overwrite")
 .saveAsTable(f"{SILVER_SCHEMA}.employee_discount_summary"))
print(f"▶ Saved {SILVER_SCHEMA}.employee_discount_summary — {emp_discounts.count():,} employees")

# ── 2b. Identify top-5 % employees by total discount amount ──────────────
threshold_row = emp_discounts.agg(
    percentile_approx("total_discount_amount", 0.95).alias("threshold")
).collect()[0]
threshold = threshold_row["threshold"] if threshold_row["threshold"] is not None else float("inf")

top_emp_ids = [r["EmployeeId"]
               for r in emp_discounts
                           .filter(col("total_discount_amount") >= threshold)
                           .select("EmployeeId")
                           .collect()]
print(f"  Top-5% discount employees (≥ {threshold:.2f}): {len(top_emp_ids)} employees")

# ── 2c. Add fraud flag (f) to the checks DataFrame ──────────────────────
if top_emp_ids:
    # Build a SQL IN-list for use with exists()
    top_emp_id_list = ",".join(str(int(e)) for e in top_emp_ids)
    checks_df = checks_df.withColumn(
        "is_fraud_top_discount_employee",
        expr(f"exists(coalesce(Discounts, array()), x -> x.EmployeeId IN ({top_emp_id_list}))")
    )
else:
    checks_df = checks_df.withColumn("is_fraud_top_discount_employee", lit(False))

print("  Added fraud flag (f): is_fraud_top_discount_employee")

# ── 2d. Write the checks Silver table ──────────────────────────────────
(checks_df.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable(f"{SILVER_SCHEMA}.checks"))
print(f"✅ Written {SILVER_SCHEMA}.checks — {checks_df.count():,} rows")

# ── 2e. Create suspect_fraud_checks table ─────────────────────────────
# Contains only checks where at least one fraud flag is true.
fraud_flag_cols = [
    "is_fraud_high_value_excess_discount",
    "is_fraud_low_value_excess_discount",
    "is_fraud_suspicious_in_store",
    "is_fraud_negative_values",
    "is_fraud_irregular_hours",
    "is_fraud_top_discount_employee",
]
fraud_condition = " OR ".join(f"{c} = true" for c in fraud_flag_cols)

spark.sql(f"""
    CREATE OR REPLACE TABLE {SILVER_SCHEMA}.suspect_fraud_checks
    AS SELECT * FROM {SILVER_SCHEMA}.checks WHERE {fraud_condition}
""")
fraud_count = spark.table(f"{SILVER_SCHEMA}.suspect_fraud_checks").count()
print(f"✅ Written {SILVER_SCHEMA}.suspect_fraud_checks — {fraud_count:,} suspect checks")

# ── 2f. Delta constraint: primary keys must never be NULL ───────────────
# Drop first if it already exists (idempotent re-runs)
spark.sql(f"ALTER TABLE {SILVER_SCHEMA}.checks DROP CONSTRAINT IF EXISTS checks_pk_not_null")
spark.sql(f"""
    ALTER TABLE {SILVER_SCHEMA}.checks
    ADD CONSTRAINT checks_pk_not_null
    CHECK (LocationId IS NOT NULL AND BusinessDate IS NOT NULL AND Id IS NOT NULL)
""")
print(f"✅ Delta constraint added: checks_pk_not_null on {SILVER_SCHEMA}.checks")

In [0]:
# ============================================================================
# 3. PROCESS ALL REMAINING BRONZE TABLES (loop)
# ============================================================================
# The `checks` table was handled separately above because it requires
# deduplication and fraud detection.  The remaining 9 tables are cleaned
# with a reusable loop: type casting → trim → drop all-null columns →
# fill numeric nulls → write to Silver.
# ============================================================================

# List all bronze tables except `checks` (already processed)
other_tables = [r.tableName for r in spark.sql(f"SHOW TABLES IN {BRONZE_SCHEMA}").collect()
                if r.tableName != "checks"]

print(f"▶ Processing {len(other_tables)} tables: {other_tables}\n")

silver_results = []  # (table_name, row_count, status)

for tbl in other_tables:
    try:
        print(f"── {tbl} ──────────────────────────────────────────")
        df = spark.table(f"{BRONZE_SCHEMA}.{tbl}")
        bronze_rows = df.count()

        # Type casting (per TYPE_CAST_RULES)
        df = apply_type_casts(df, tbl)

        # Trim whitespace on string columns
        df = trim_string_columns(df)

        # Drop columns that are 100 % NULL
        df = drop_all_null_columns(df, tbl)

        # Fill numeric nulls (< 5 % missing) with 0
        df = fill_numeric_nulls(df, tbl)

        # Write to Silver
        (df.write.format("delta").mode("overwrite")
         .option("overwriteSchema", "true")
         .saveAsTable(f"{SILVER_SCHEMA}.{tbl}"))

        silver_rows = spark.table(f"{SILVER_SCHEMA}.{tbl}").count()
        print(f"  ✅ {SILVER_SCHEMA}.{tbl} — {silver_rows:,} rows (from {bronze_rows:,} bronze rows)\n")
        silver_results.append((tbl, silver_rows, "OK"))

    except Exception as e:
        print(f"  ❌ {tbl} FAILED: {e}\n")
        silver_results.append((tbl, 0, str(e)))

# Also record the checks table result
checks_silver_count = spark.table(f"{SILVER_SCHEMA}.checks").count()
silver_results.insert(0, ("checks", checks_silver_count, "OK"))

# Record fraud-specific tables
fraud_count = spark.table(f"{SILVER_SCHEMA}.suspect_fraud_checks").count()
silver_results.append(("suspect_fraud_checks", fraud_count, "OK"))
emp_summary_count = spark.table(f"{SILVER_SCHEMA}.employee_discount_summary").count()
silver_results.append(("employee_discount_summary", emp_summary_count, "OK"))
quarantine_count = spark.table(f"{SILVER_SCHEMA}.quarantine_bad_data").count()
silver_results.append(("quarantine_bad_data", quarantine_count, "OK"))

In [0]:
# ============================================================================
# 4. SILVER LAYER SUMMARY
# ============================================================================

print("=" * 70)
print("Silver layer ingestion summary")
print("=" * 70)
for tbl_name, rows, status in silver_results:
    print(f"  {SILVER_SCHEMA}.{tbl_name:<40} {rows:>10,}  {status}")
print("=" * 70)

# ── Fraud flag breakdown ────────────────────────────────────────────────
print("\nFraud flag breakdown (checks table):")
print("-" * 70)
fraud_stats = spark.table(f"{SILVER_SCHEMA}.checks").agg(
    _count(when(col("is_fraud_high_value_excess_discount"), 1)).alias("high_value_excess_discount"),
    _count(when(col("is_fraud_low_value_excess_discount"), 1)).alias("low_value_excess_discount"),
    _count(when(col("is_fraud_suspicious_in_store"), 1)).alias("suspicious_in_store"),
    _count(when(col("is_fraud_negative_values"), 1)).alias("negative_values"),
    _count(when(col("is_fraud_irregular_hours"), 1)).alias("irregular_hours"),
    _count(when(col("is_fraud_top_discount_employee"), 1)).alias("top_discount_employee"),
).collect()[0]

flag_labels = [
    ("(a) High-value + excess discount",   "high_value_excess_discount"),
    ("(b) Low-value + excess discount",    "low_value_excess_discount"),
    ("(c) Suspicious in-store",           "suspicious_in_store"),
    ("(d) Negative tips/prices",          "negative_values"),
    ("(e) Irregular hours (2-7 AM)",      "irregular_hours"),
    ("(f) Top discount employee",         "top_discount_employee"),
]
for label, key in flag_labels:
    print(f"  {label:<35} {fraud_stats[key]:>8,} checks")
print("-" * 70)
total_fraud = spark.table(f"{SILVER_SCHEMA}.suspect_fraud_checks").count()
total_checks = spark.table(f"{SILVER_SCHEMA}.checks").count()
pct = (total_fraud / total_checks * 100) if total_checks > 0 else 0
print(f"  {'TOTAL suspect checks':<35} {total_fraud:>8,} checks ({pct:.1f}% of {total_checks:,})")
print("-" * 70)

# ── Preview suspect fraud checks ────────────────────────────────────────
print("\nSample suspect fraud checks:")
display(spark.table(f"{SILVER_SCHEMA}.suspect_fraud_checks")
        .select("LocationId", "BusinessDate", "Id", "total_payment_amount",
                "total_discount_amount", "discount_ratio", "open_hour",
                "is_fraud_high_value_excess_discount", "is_fraud_irregular_hours")
        .limit(10))